# PG-LIF -- P4: Sensitivity Analysis
**Feeds:** manuscript Section 8 (the `[[RESULT: sensitivity analysis]]` marker) and supports the Discussion's
point that the extra hyperparameters PG-LIF introduces are not fragile.

**What this does.** Trains PG-LIF on SHD while sweeping each of its extra hyperparameters -- dendritic
threshold theta_d, plateau amplitude P0, plateau time constant tau_p, plateau refractory tref_p, and the
somatic coupling kappa (learnable by default, here also swept as a fixed init) -- ONE at a time around its
default, holding the others at the values used in the main sweep. Reports best SHD test accuracy vs. each
swept value, so the manuscript can state how flat (or not) each dependence is.

**Design consistency.** Uses the SAME PG-LIF cell, two-layer 700-128-128-20 architecture, binning, readout,
optimizer (Adam), schedule (MultiStepLR[40,80]x0.1), LR (5e-4), and dropout (0.1) as the final P1 sweep
(`paper_mode_final`), so these numbers are directly comparable to the main PG-LIF result. Reuses the SHD
cache in `My Drive/PG_LIF/data/SHD/`.

**Cost.** Each point is one full PG-LIF training run (~100 epochs, early-stopped). Defaults keep the grids
small (3-4 points per hyperparameter, ~17 runs total). This is a lot of GPU time -- run it AFTER the main
sweep, over one or more sessions; it is fully resumable (every point writes a JSON and is skipped on rerun).
To shorten it, reduce `EPOCHS` to ~40 for a coarser but faster read, or trim the grids.

In [ ]:
import os, json, time, math
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    if not os.path.isdir(ROOT):
        raise RuntimeError("Drive mounted but no My Drive/MyDrive under /content/drive - re-run and approve.")
    BASE = os.path.join(ROOT, 'PG_LIF'); IN_COLAB = True
except ImportError:
    print('WARNING: not in Colab - local non-persistent folder.'); BASE = './PG_LIF'
except Exception as e:
    raise RuntimeError(f'Drive did not mount ({e}). Fix the mount before running so results persist.') from e
DATA = os.path.join(BASE, 'data', 'SHD')
OUT = os.path.join(BASE, 'P4_results', 'sensitivity')
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

In [ ]:
# --- CONFIG (defaults match paper_mode_final) ---
T_BINS, N_IN, N_OUT, HIDDEN = 250, 700, 20, 128
BATCH, LR, MAX_TIME, EPOCHS = 64, 5e-4, 1.4, 100
SCHEDULE = [40, 80]; DROPOUT = 0.1
SEED = 0
EARLY_STOP_PATIENCE, EARLY_STOP_MIN_EPOCH = 15, 45

# PG-LIF defaults (the point every sweep passes through)
DEF = dict(theta_d=1.0, P0=1.0, tau_p=T_BINS/2, tref_p=10, kappa0=1.0)

# Sweep grids: each maps a hyperparameter to the values tried (default value included in each).
GRIDS = {
    'theta_d': [0.5, 1.0, 1.5, 2.0],
    'P0':      [0.5, 1.0, 2.0],
    'tau_p':   [T_BINS/4, T_BINS/2, T_BINS],       # 62.5, 125, 250 steps
    'tref_p':  [0, 10, 20],
    'kappa0':  [0.5, 1.0, 2.0],                      # fixed kappa init (learnable in main run; here probed)
}
print('Total runs this config:', sum(len(v) for v in GRIDS.values()),
      '(default point is re-used across sweeps, not retrained each time)')
V_CLAMP, P_CLAMP = 20.0, 50.0   # v9: forward-pass state clamps for the PG-LIF family (match main sweep)

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn
def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
# download if missing (same as main notebook)
import gzip, shutil, urllib.request
for name, url in {'shd_train.h5':'https://zenkelab.org/datasets/shd_train.h5.gz',
                  'shd_test.h5':'https://zenkelab.org/datasets/shd_test.h5.gz'}.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst+'.gz'; print('downloading', url); urllib.request.urlretrieve(url, gz)
        with gzip.open(gz,'rb') as fi, open(dst,'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)
TR_raw = load_split('shd_train.h5'); TE_raw = load_split('shd_test.h5')

def precompute_dense(split):
    times, units, labels = split; n = len(labels)
    time_bins = np.linspace(0, MAX_TIME, num=T_BINS)     # official digitize/linspace binning
    X = torch.zeros(n, T_BINS, N_IN, dtype=torch.bool)
    for i in range(n):
        tb = np.clip(np.digitize(times[i], time_bins), 0, T_BINS-1)
        X[i, tb, units[i]] = True
    return X, torch.as_tensor(labels, dtype=torch.long)
TR_X, TR_Y = precompute_dense(TR_raw); TE_X, TE_Y = precompute_dense(TE_raw)
print('train', len(TR_Y), 'test', len(TE_Y))

def batches(X, Y, bs, shuffle, device, drop_last=False):
    n = len(Y); idx = torch.randperm(n) if shuffle else torch.arange(n)
    if drop_last: n = (n//bs)*bs
    for b0 in range(0, n, bs):
        sel = idx[b0:b0+bs]; yield X[sel].float().to(device), Y[sel].to(device)

In [ ]:
class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x): ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs()/Triangle.gamma, min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0/tau)

class PGLIFCell(nn.Module):
    """Same PG-LIF cell as the main sweep (manuscript Eqs. 6-10, scaled drive kappa*p*(1-am))."""
    th = 1.0; beta = 1.0
    def __init__(self, N, theta_d=1.0, P0=1.0, tau_p=None, tref_p=10, kappa0=1.0):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(tau_p or T_BINS/2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0/(1-ap0))))
        self.kappa = nn.Parameter(torch.full((N,), float(kappa0)))
        self.P0, self.tref_p = P0, tref_p
        self.register_buffer('theta_d', torch.full((N,), float(theta_d)))
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z(); self.rp = z()
    def forward(self, I_ff, I_rec):
        self.vd = self.ad*self.vd + I_ff
        self.vd = torch.clamp(self.vd, -V_CLAMP, V_CLAMP)   # v9: match main sweep (forward-pass safeguard)
        ed = spike_fn(self.vd - self.theta_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp-1, min=0) + ed.detach()*self.tref_p
        self.p = torch.sigmoid(self.ap_logit)*self.p + self.P0*ed
        self.p = torch.clamp(self.p, max=P_CLAMP)
        self.vs = self.am*self.vs + I_ff + I_rec + self.kappa*self.p*(1-self.am)
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)
        th = self.th + self.beta*self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach()*th.detach(); self.a = self.aa*self.a + s.detach()
        return s

class RecLayer(nn.Module):
    def __init__(self, n_in, n_hid, **kw):
        super().__init__()
        self.w_in = nn.Linear(n_in, n_hid); self.drop = nn.Dropout(DROPOUT)
        self.w_rec = nn.Linear(n_hid, n_hid, bias=True)   # match main sweep: bias=True, default init
        self.cell = PGLIFCell(n_hid, **kw); self.n_hid = n_hid
    def init(self, B, dev): self.cell.init(B, dev); self.s = torch.zeros(B, self.n_hid, device=dev)
    def step(self, x_t):
        return (lambda s: s)(self._step(x_t))
    def _step(self, x_t):
        iff = self.drop(self.w_in(x_t)); irec = self.w_rec(self.s)
        self.s = self.cell(iff, irec); return self.s

class RecSNN(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.layer1 = RecLayer(N_IN, HIDDEN, **kw); self.layer2 = RecLayer(HIDDEN, HIDDEN, **kw)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
    def forward(self, x):
        B, T, _ = x.shape; dev = x.device
        self.layer1.init(B, dev); self.layer2.init(B, dev)
        out = torch.zeros(B, N_OUT, device=dev)
        for t in range(T):
            out = out + self.w_out(self.layer2.step(self.layer1.step(x[:, t])))
        return out

def accuracy(model, device):
    model.eval(); c = t = 0
    with torch.no_grad():
        for x, y in batches(TE_X, TE_Y, 128, False, device):
            c += (model(x).argmax(1) == y).sum().item(); t += len(y)
    return c/t

def train_point(name, kw, device):
    f = os.path.join(OUT, name + '.json')
    if os.path.exists(f): print('[skip]', name); return json.load(open(f))
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = RecSNN(**kw).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)
    crit = nn.CrossEntropyLoss(); best, since = 0.0, 0
    for ep in range(EPOCHS):
        m.train()
        n_skip = 0; n_tot = 0
        for x, y in batches(TR_X, TR_Y, BATCH, True, device, drop_last=True):
            opt.zero_grad(); loss = crit(m(x), y); loss.backward()
            # v9: discard any step whose gradient is non-finite (BPTT exploding-gradient guard)
            if not all(p.grad is None or torch.isfinite(p.grad).all() for p in m.parameters()):
                n_skip += 1; opt.zero_grad(); n_tot += 1; continue
            torch.nn.utils.clip_grad_value_(m.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step()
            n_tot += 1
        sch.step(); acc = accuracy(m, device)
        if acc > best + 1e-4: best, since = acc, 0
        else: since += 1
        if EARLY_STOP_PATIENCE and since >= EARLY_STOP_PATIENCE and ep >= EARLY_STOP_MIN_EPOCH:
            break
    res = {'name': name, 'kw': {k: str(v) for k, v in kw.items()}, 'best_test_acc': best, 'last_epoch_skip_rate': (n_skip/n_tot if n_tot else 0.0)}
    json.dump(res, open(f, 'w'), indent=2); print(f'{name}: {best:.4f}')
    return res

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU - this will be very slow.')
# default point (shared reference for all sweeps)
default_res = train_point('default', dict(theta_d=DEF['theta_d'], P0=DEF['P0'], tau_p=DEF['tau_p'],
                                          tref_p=DEF['tref_p'], kappa0=DEF['kappa0']), device)
RESULTS = {'default': default_res['best_test_acc']}
for hp, grid in GRIDS.items():
    for v in grid:
        if v == DEF[hp]:
            RESULTS.setdefault(hp, {})[v] = default_res['best_test_acc']; continue
        kw = {k: DEF[k] for k in DEF}; kw[hp] = v
        name = f'{hp}_{v}'
        r = train_point(name, dict(theta_d=kw['theta_d'], P0=kw['P0'], tau_p=kw['tau_p'],
                                   tref_p=kw['tref_p'], kappa0=kw['kappa0']), device)
        RESULTS.setdefault(hp, {})[v] = r['best_test_acc']
json.dump(RESULTS, open(os.path.join(OUT, 'sensitivity_summary.json'), 'w'), indent=2, default=str)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(GRIDS), figsize=(4*len(GRIDS), 3.5), sharey=True)
for ax, (hp, grid) in zip(axes, GRIDS.items()):
    xs = list(grid); ys = [RESULTS[hp][v] for v in xs]
    ax.plot(range(len(xs)), [y*100 for y in ys], 'o-')
    ax.set_xticks(range(len(xs))); ax.set_xticklabels([str(round(float(v),1)) for v in xs], rotation=45)
    ax.axhline(RESULTS['default']*100, color='k', ls=':', lw=0.8)
    ax.set_title(hp); ax.set_xlabel('value')
axes[0].set_ylabel('SHD test acc (%)')
plt.suptitle('PG-LIF sensitivity to its extra hyperparameters (dotted = default)')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_sensitivity.png'), dpi=300); plt.show()
print()
for hp, grid in GRIDS.items():
    vals = [RESULTS[hp][v]*100 for v in grid]
    print(f'{hp:10s} range {min(vals):.1f}-{max(vals):.1f}% (spread {max(vals)-min(vals):.1f}pp) '
          f'across {[round(float(v),1) for v in grid]}')
print('\nInterpretation: small spread = robust to that hyperparameter; large spread = sensitive, needs care.')

### Using these for the manuscript
Replace the `[[RESULT: sensitivity analysis]]` marker in Section 8 with a sentence like: "PG-LIF is robust to
its additional hyperparameters over the ranges tested (accuracy spread < X pp for theta_d, P0, tref_p, kappa;
the plateau time constant tau_p is the most influential, with Y pp spread)." Fill X and Y from the printed
spreads. If any hyperparameter shows a large spread, report it honestly as the one requiring tuning, which is
consistent with the Section 4.3 conditioning discussion for kappa/scaling.